[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/098.%20The%20Green%20Vehicle%20Routing%20Problem/P98-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 98. The Green Vehicle Routing Problem

## Tier 4: Tabular Q-Learning Agent

### Purpose and scope

This notebook models a small electric-vehicle routing problem as a Markov decision process (MDP) and trains a tabular Q-learning agent. The agent repeatedly chooses a customer, a charging station, or the depot using the information in its current state. The example is deliberately small enough for a table of state-action values; it does not implement a neural network or a Deep Q-Network.

The inputs are a distance matrix, an emissions matrix, customer demands, vehicle capacity, and battery parameters. The outputs are the learned Q-table, training and evaluation statistics, a sample route, and diagnostic plots. A random valid-action policy is included as a baseline. The training and evaluation use the same fixed instance, so evaluation measures performance on this instance rather than generalization to new instances.

### Model assumptions

- Node `0` is the depot, nodes `1` through `4` are customers, and the final node is a charging station.
- A vehicle may make multiple depot trips. `current_load` records the demand already served since its most recent visit to the depot; it is the used capacity budget for that trip. Returning to the depot resets this budget. This represents reloading for another trip without modeling individual parcels or multiple vehicles.
- Traveling from node \(i\) to node \(j\) consumes distance-proportional battery and incurs the corresponding entries in the distance and emissions matrices. Arriving at the charging station restores the battery to its maximum level.
- A customer can be served only once. A successful episode serves every customer and then returns to the depot. Episodes are capped at 20 actions to ensure that training and evaluation terminate.
- The reward is a teaching objective combining distance, emissions, a service reward, and fixed rewards or penalties for charging, depot returns, and episode termination. It is not a direct monetary cost and does not guarantee that minimizing reward is identical to minimizing distance or emissions separately.
- Travel times, time windows, stochastic demand, traffic, multiple vehicles, and uncertainty are outside this example.

### From a routing problem to an MDP

An MDP describes sequential decisions using states, actions, transition rules, and rewards. The state must contain enough information to determine which actions are feasible and what happens next. Here it is

$$
s = (\ell, b, M, u),
$$

where \(\ell\) is the current node, \(b\) is the remaining battery, \(M\) is a bitmask of unvisited customers, and \(u\) is the capacity used since the last depot visit. The bitmask has one bit per customer: a bit is 1 when that customer remains to be served. Including \(u\) is necessary because capacity-feasible actions depend on how much demand has already been served on the current trip.

Actions are node identifiers. At each state, the environment supplies only feasible actions: reachable unvisited customers that fit within the remaining trip capacity, a reachable charging station, and a reachable depot when a return or capacity reset is needed. The transition updates location, battery, remaining customers, and used capacity. The reward for visiting customer \(j\) from \(\ell\) is

$$
r(s,j) = 10 - 0.5\,d_{\ell j} - 0.5\,e_{\ell j},
$$

where \(d_{\ell j}\) is distance and \(e_{\ell j}\) is emissions. Charging, depot returns, and successful completion have their own rewards in the environment code. The matrices provide the units used by the example; the weights make the two travel measures contribute equally by numerical value, not necessarily by physical unit.

### Q-learning update

Q-learning estimates the long-run value of taking action \(a\) in state \(s\). After observing reward \(r\) and next state \(s'\), it updates the estimate as

$$
Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma\max_{a'\in A(s')}Q(s',a') - Q(s,a)\right].
$$

Here \(\alpha\) is the learning rate, \(\gamma\) discounts future rewards, and \(A(s')\) is the set of feasible next actions. For a terminal transition, the future-value term is zero. The code implements this rule in `QLearningAgent.update`. An \(\varepsilon\)-greedy policy explores by choosing a random feasible action with probability \(\varepsilon\); otherwise it chooses a feasible action with the largest learned Q-value.

The full state space grows rapidly with the number of customers: the remaining-customer component alone has \(2^n\) possible subsets for \(n\) customers. Battery and capacity levels add further dimensions. Tabular Q-learning is therefore suitable here as a small-instance teaching method, not as a scalable solution for large routing problems. Learning curves and evaluation results are empirical diagnostics; they do not prove optimality or convergence.

### Learning workflow

1. Define the environment and its feasible actions.
2. Create the fixed small instance and inspect its node and demand data.
3. Train the tabular Q-learning agent for 1,000 episodes with an \(\varepsilon\)-greedy policy.
4. Evaluate the learned policy and a random valid-action baseline over 100 episodes each.
5. Inspect a learned-policy episode, learning statistics, and diagnostic plots.


In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# A fixed seed makes the exploration and random baseline reproducible.
np.random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class GreenVRPEnvironment:
    """Environment for a small electric-vehicle routing problem."""

    def __init__(
        self,
        distances,
        emissions,
        demands,
        vehicle_capacity,
        max_battery=100,
        battery_consumption_rate=0.5,
    ):
        """Initialize node data, resource limits, and the initial state."""
        self.distances = np.asarray(distances, dtype=float)
        self.emissions = np.asarray(emissions, dtype=float)
        self.demands = dict(demands)
        self.vehicle_capacity = vehicle_capacity
        self.max_battery = max_battery
        self.battery_consumption_rate = battery_consumption_rate

        if self.distances.ndim != 2 or self.distances.shape[0] != self.distances.shape[1]:
            raise ValueError("The distance matrix must be square.")
        if self.emissions.shape != self.distances.shape:
            raise ValueError("Distance and emissions matrices must have the same shape.")
        if np.any(self.distances < 0) or np.any(self.emissions < 0):
            raise ValueError("Distances and emissions must be nonnegative.")
        if max_battery < 0 or battery_consumption_rate < 0 or vehicle_capacity < 0:
            raise ValueError("Battery, consumption rate, and capacity must be nonnegative.")

        self.n_nodes = self.distances.shape[0]
        self.depot = 0
        self.charging_station = self.n_nodes - 1
        self.customers = list(range(1, self.charging_station))

        if set(self.demands) != set(self.customers):
            raise ValueError("Demand data must contain exactly one entry per customer.")
        if any(demand < 0 for demand in self.demands.values()):
            raise ValueError("Customer demands must be nonnegative.")

        # This count is a nominal upper bound for integer-valued battery and capacity units.
        self.n_states = (
            self.n_nodes
            * (int(max_battery) + 1)
            * (2 ** len(self.customers))
            * (int(vehicle_capacity) + 1)
        )
        # Actions are node identifiers, including the depot and charging station.
        self.n_actions = self.n_nodes
        self.reset()

    def reset(self):
        """Start a new episode at the depot with a full battery."""
        self.current_location = self.depot
        self.battery_level = self.max_battery
        self.remaining_customers = set(self.customers)
        self.current_load = 0
        self.total_distance = 0.0
        self.total_emissions = 0.0
        self.episode_steps = 0
        return self.get_state()

    def get_state(self):
        """Return location, battery, unvisited-customer mask, and used capacity."""
        customer_bitmask = 0
        for customer in self.remaining_customers:
            customer_bitmask |= 1 << (customer - 1)
        return (
            self.current_location,
            self.battery_level,
            customer_bitmask,
            self.current_load,
        )

    def get_valid_actions(self, state):
        """Return reachable actions that respect service and trip-capacity rules."""
        current_location, battery_level, remaining_mask, current_load = state
        remaining_customers = {
            customer
            for customer in self.customers
            if remaining_mask & (1 << (customer - 1))
        }
        valid_actions = []

        for customer in self.customers:
            if customer not in remaining_customers:
                continue
            distance = self.distances[current_location, customer]
            battery_needed = distance * self.battery_consumption_rate
            demand_needed = self.demands[customer]
            if (
                battery_level + 1e-9 >= battery_needed
                and current_load + demand_needed <= self.vehicle_capacity + 1e-9
            ):
                valid_actions.append(customer)

        station_distance = self.distances[current_location, self.charging_station]
        station_battery = station_distance * self.battery_consumption_rate
        if (
            current_location != self.charging_station
            and battery_level + 1e-9 >= station_battery
        ):
            valid_actions.append(self.charging_station)

        depot_distance = self.distances[current_location, self.depot]
        depot_battery = depot_distance * self.battery_consumption_rate
        can_reach_depot = battery_level + 1e-9 >= depot_battery
        all_customers_served = not remaining_customers
        capacity_reset_needed = (
            current_location == self.depot
            and current_load > 0
            and not valid_actions
        )
        if can_reach_depot and (
            all_customers_served
            or current_location != self.depot
            or capacity_reset_needed
        ):
            valid_actions.append(self.depot)

        return valid_actions

    def step(self, action):
        """Apply a feasible action and return the next state, reward, and terminal flag."""
        if action not in self.get_valid_actions(self.get_state()):
            return self.get_state(), -100.0, True

        origin = self.current_location
        distance = self.distances[origin, action]
        emissions = self.emissions[origin, action]
        battery_consumed = distance * self.battery_consumption_rate

        self.current_location = action
        self.battery_level = max(0.0, self.battery_level - battery_consumed)
        self.total_distance += distance
        self.total_emissions += emissions
        self.episode_steps += 1
        done = False

        if action == self.charging_station:
            self.battery_level = self.max_battery
            reward = -5.0
        elif action == self.depot:
            if not self.remaining_customers:
                reward = 50.0
                done = True
            else:
                reward = -20.0
            self.current_load = 0
        else:
            self.current_load += self.demands[action]
            self.remaining_customers.remove(action)
            reward = -(0.5 * distance + 0.5 * emissions) + 10.0

        # A stranded vehicle cannot continue, so the transition ends with a penalty.
        if not done and not self.get_valid_actions(self.get_state()):
            reward -= 50.0
            done = True

        if self.episode_steps >= 20 and not done:
            reward -= 50.0
            done = True

        return self.get_state(), reward, done

    def render(self, mode="text"):
        """Print the current route state and cumulative travel measures."""
        if mode == "text":
            print(f"Location: {self.current_location}, Battery: {self.battery_level:.1f}")
            print(f"Remaining customers: {sorted(self.remaining_customers)}")
            print(f"Capacity used on current trip: {self.current_load:.1f}")
            print(
                f"Total distance: {self.total_distance:.1f}, "
                f"Total emissions: {self.total_emissions:.1f}"
            )


class QLearningAgent:
    """Tabular Q-learning agent with an epsilon-greedy action policy."""

    def __init__(
        self,
        n_states,
        n_actions,
        learning_rate=0.1,
        discount_factor=0.95,
        epsilon=0.1,
        epsilon_decay=0.995,
        epsilon_min=0.01,
    ):
        self.n_states = n_states
        self.n_actions = n_actions
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.q_table = defaultdict(lambda: defaultdict(float))
        self.training_history = {}

    def _state_to_key(self, state):
        """Use the complete state tuple as a stable dictionary key."""
        return tuple(state)

    def get_action(self, state, valid_actions, training=True):
        """Choose a feasible action by epsilon-greedy exploration or exploitation."""
        if not valid_actions:
            raise ValueError("The state has no feasible actions.")

        state_key = self._state_to_key(state)
        if training and np.random.random() < self.epsilon:
            return int(np.random.choice(valid_actions))

        q_values = self.q_table[state_key]
        best_q_value = max(q_values[action] for action in valid_actions)
        best_actions = [
            action for action in valid_actions
            if np.isclose(q_values[action], best_q_value)
        ]
        return int(np.random.choice(best_actions))

    def update(self, state, action, reward, next_state, next_valid_actions, done):
        """Apply the one-step Q-learning update, omitting future value at termination."""
        state_key = self._state_to_key(state)
        next_state_key = self._state_to_key(next_state)
        current_q = self.q_table[state_key][action]

        if done:
            max_next_q = 0.0
        else:
            if not next_valid_actions:
                raise ValueError("A nonterminal state must have a feasible action.")
            next_q_values = self.q_table[next_state_key]
            max_next_q = max(next_q_values[a] for a in next_valid_actions)

        target = reward + self.discount_factor * max_next_q
        self.q_table[state_key][action] = current_q + self.learning_rate * (target - current_q)

    def decay_epsilon(self):
        """Reduce exploration gradually while retaining a minimum exploration rate."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def train(self, env, n_episodes=1000):
        """Train on episodes generated by the supplied environment."""
        episode_rewards = []
        episode_steps = []
        epsilon_history = []

        print("Starting Q-learning training...")
        print(f"Number of episodes: {n_episodes}")
        print(f"Initial epsilon: {self.epsilon:.3f}")

        for episode in range(n_episodes):
            state = env.reset()
            total_reward = 0.0
            steps = 0
            done = False

            while not done:
                valid_actions = env.get_valid_actions(state)
                action = self.get_action(state, valid_actions, training=True)
                next_state, reward, done = env.step(action)
                next_valid_actions = env.get_valid_actions(next_state)
                self.update(
                    state,
                    action,
                    reward,
                    next_state,
                    next_valid_actions,
                    done,
                )
                state = next_state
                total_reward += reward
                steps += 1

            episode_rewards.append(total_reward)
            episode_steps.append(steps)
            epsilon_history.append(self.epsilon)
            self.decay_epsilon()

            if (episode + 1) % 100 == 0:
                average_reward = np.mean(episode_rewards[-100:])
                print(
                    f"Episode {episode + 1:4d}: "
                    f"mean reward over the last 100 episodes = {average_reward:.2f}, "
                    f"epsilon = {self.epsilon:.3f}"
                )

        self.training_history = {
            "episode_rewards": episode_rewards,
            "episode_steps": episode_steps,
            "epsilon_history": epsilon_history,
        }
        print("Training completed.")
        print(f"Final epsilon: {self.epsilon:.3f}")
        print(f"Mean reward over the last 100 episodes: {np.mean(episode_rewards[-100:]):.2f}")
        return self.training_history

    def evaluate(self, env, n_episodes=100):
        """Evaluate the greedy policy without changing the learned Q-table."""
        eval_rewards = []
        eval_distances = []
        eval_emissions = []
        eval_success = []

        print(f"Evaluating the agent for {n_episodes} episodes...")
        for _ in range(n_episodes):
            state = env.reset()
            total_reward = 0.0
            done = False

            while not done:
                valid_actions = env.get_valid_actions(state)
                action = self.get_action(state, valid_actions, training=False)
                state, reward, done = env.step(action)
                total_reward += reward

            eval_rewards.append(total_reward)
            eval_distances.append(env.total_distance)
            eval_emissions.append(env.total_emissions)
            eval_success.append(not env.remaining_customers)

        results = {
            "mean_reward": np.mean(eval_rewards),
            "std_reward": np.std(eval_rewards),
            "mean_distance": np.mean(eval_distances),
            "std_distance": np.std(eval_distances),
            "mean_emissions": np.mean(eval_emissions),
            "std_emissions": np.std(eval_emissions),
            "success_rate": np.mean(eval_success),
        }
        print("Evaluation results:")
        print(f"  Mean reward: {results['mean_reward']:.2f} +/- {results['std_reward']:.2f}")
        print(f"  Mean distance: {results['mean_distance']:.2f} +/- {results['std_distance']:.2f} distance units")
        print(f"  Mean emissions: {results['mean_emissions']:.2f} +/- {results['std_emissions']:.2f} emissions units")
        print(f"  Success rate: {results['success_rate']:.2%}")
        return results

In [ ]:
# Nodes: 0 is the depot, 1-4 are customers, and 5 is the charging station.
distances = [
    [0, 10, 15, 12, 18, 8],
    [10, 0, 8, 14, 12, 15],
    [15, 8, 0, 10, 6, 12],
    [12, 14, 10, 0, 8, 10],
    [18, 12, 6, 8, 0, 14],
    [8, 15, 12, 10, 14, 0],
]

emissions = [
    [0, 8, 12, 10, 15, 3],
    [8, 0, 6, 12, 10, 12],
    [12, 6, 0, 8, 5, 10],
    [10, 12, 8, 0, 6, 8],
    [15, 10, 5, 6, 0, 11],
    [3, 12, 10, 8, 11, 0],
]

demands = {1: 5, 2: 8, 3: 6, 4: 7}
vehicle_capacity = 15

print("Green VRP instance")
print("=" * 40)
print(f"Number of customers: {len(demands)}")
print(f"Vehicle capacity per depot trip: {vehicle_capacity} units")
print(f"Total demand: {sum(demands.values())} units")
print("Maximum battery: 100 units")
print("Battery consumption rate: 0.5 units per distance unit")

print("\nCustomer demands:")
for customer, demand in demands.items():
    print(f"  Customer {customer}: {demand} units")

print("\nNode mapping:")
print("  0: Depot")
for customer in demands:
    print(f"  {customer}: Customer {customer}")
print(f"  {len(distances) - 1}: Charging station")

In [ ]:
print("=" * 60)
print("REINFORCEMENT LEARNING SETUP")
print("=" * 60)

env = GreenVRPEnvironment(
    distances=distances,
    emissions=emissions,
    demands=demands,
    vehicle_capacity=vehicle_capacity,
    max_battery=100,
    battery_consumption_rate=0.5,
)

print("Environment created:")
print(f"  Nominal state-space upper bound: {env.n_states}")
print(f"  Number of node-based actions: {env.n_actions}")
print(f"  Customers: {env.customers}")
print(f"  Charging station: {env.charging_station}")

agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    learning_rate=0.1,
    discount_factor=0.95,
    epsilon=0.3,
    epsilon_decay=0.995,
    epsilon_min=0.01,
)

print("\nAgent parameters:")
print(f"  Learning rate: {agent.learning_rate}")
print(f"  Discount factor: {agent.discount_factor}")
print(f"  Initial epsilon: {agent.epsilon}")
print(f"  Epsilon decay: {agent.epsilon_decay}")
print(f"  Minimum epsilon: {agent.epsilon_min}")

print("\nTesting the environment with random valid actions:")
state = env.reset()
print(f"Initial state (location, battery, customer mask, used capacity): {state}")

for step in range(5):
    valid_actions = env.get_valid_actions(state)
    action = int(np.random.choice(valid_actions))
    print(f"Step {step + 1}: feasible actions = {valid_actions}; selected node = {action}")
    state, reward, done = env.step(action)
    print(f"  Reward: {reward:.2f}; terminal: {done}")
    print(f"  Next state: {state}")
    if done:
        break

print("\nEnvironment state after the test:")
env.render()

In [ ]:
print("\n" + "=" * 60)
print("Q-LEARNING TRAINING")
print("=" * 60)

training_history = agent.train(env, n_episodes=1000)

print(f"Q-table states visited: {len(agent.q_table)}")
print(f"Final epsilon: {agent.epsilon:.4f}")

In [ ]:
print("\n" + "=" * 60)
print("AGENT EVALUATION")
print("=" * 60)

eval_results = agent.evaluate(env, n_episodes=100)

def random_policy_evaluation(env, n_episodes=100):
    """Measure a baseline that samples uniformly from feasible actions."""
    rewards = []
    distances = []
    emissions_totals = []
    successes = []

    for _ in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False

        while not done:
            valid_actions = env.get_valid_actions(state)
            action = int(np.random.choice(valid_actions))
            state, reward, done = env.step(action)
            total_reward += reward

        rewards.append(total_reward)
        distances.append(env.total_distance)
        emissions_totals.append(env.total_emissions)
        successes.append(not env.remaining_customers)

    return {
        "mean_reward": np.mean(rewards),
        "mean_distance": np.mean(distances),
        "mean_emissions": np.mean(emissions_totals),
        "success_rate": np.mean(successes),
    }


print("\nRandom valid-action baseline")
random_results = random_policy_evaluation(env, n_episodes=100)
print(f"  Mean reward: {random_results['mean_reward']:.2f}")
print(f"  Mean distance: {random_results['mean_distance']:.2f} distance units")
print(f"  Mean emissions: {random_results['mean_emissions']:.2f} emissions units")
print(f"  Success rate: {random_results['success_rate']:.2%}")

print("\nQ-learning minus random-baseline differences")
print(f"  Mean reward difference: {eval_results['mean_reward'] - random_results['mean_reward']:+.2f}")
print(f"  Mean distance difference: {eval_results['mean_distance'] - random_results['mean_distance']:+.2f} distance units")
print(f"  Mean emissions difference: {eval_results['mean_emissions'] - random_results['mean_emissions']:+.2f} emissions units")
print(f"  Success-rate difference: {(eval_results['success_rate'] - random_results['success_rate']) * 100:+.1f} percentage points")

In [ ]:
print("\n" + "=" * 60)
print("LEARNED POLICY DEMONSTRATION")
print("=" * 60)

state = env.reset()
episode_log = []
done = False
print("Starting an episode with the greedy learned policy.")
print(f"Initial state: {state}")

while not done:
    valid_actions = env.get_valid_actions(state)
    action = agent.get_action(state, valid_actions, training=False)
    state_key = agent._state_to_key(state)
    q_value = agent.q_table[state_key][action]
    next_state, reward, done = env.step(action)
    episode_log.append(
        {
            "state": state,
            "valid_actions": valid_actions,
            "action": action,
            "q_value": q_value,
            "reward": reward,
            "next_state": next_state,
            "done": done,
        }
    )

    if action == env.depot:
        action_name = "Depot"
    elif action == env.charging_station:
        action_name = "Charging station"
    else:
        action_name = f"Customer {action}"

    print(f"\nStep {len(episode_log)}: {action_name} (Q-value: {q_value:.2f})")
    print(f"  State: {state}")
    print(f"  Feasible actions: {valid_actions}")
    print(f"  Reward: {reward:.2f}")
    print(f"  Next state: {next_state}")
    state = next_state

print(f"\nEpisode ended after {len(episode_log)} actions.")
print(f"Total distance: {env.total_distance:.2f} distance units")
print(f"Total emissions: {env.total_emissions:.2f} emissions units")
print(f"Total reward: {sum(step['reward'] for step in episode_log):.2f}")
print(f"All customers served: {not env.remaining_customers}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Tabular Q-Learning Diagnostics for the Green VRP", fontsize=16, fontweight="bold")

episodes = np.arange(1, len(training_history["episode_rewards"]) + 1)
window_size = 50

ax1 = axes[0, 0]
ax1.plot(episodes, training_history["episode_rewards"], alpha=0.55, color="tab:blue", linewidth=1)
reward_average = pd.Series(training_history["episode_rewards"]).rolling(window_size).mean()
ax1.plot(episodes, reward_average, color="tab:red", linewidth=2, label=f"Rolling mean ({window_size} episodes)")
ax1.set_xlabel("Episode")
ax1.set_ylabel("Episode reward (reward units)")
ax1.set_title("Training rewards")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(episodes, training_history["episode_steps"], alpha=0.55, color="tab:green", linewidth=1)
step_average = pd.Series(training_history["episode_steps"]).rolling(window_size).mean()
ax2.plot(episodes, step_average, color="tab:orange", linewidth=2, label=f"Rolling mean ({window_size} episodes)")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Actions per episode")
ax2.set_title("Episode length")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
ax3.plot(episodes, training_history["epsilon_history"], color="tab:purple", linewidth=2)
ax3.set_xlabel("Episode")
ax3.set_ylabel("Exploration probability, epsilon")
ax3.set_title("Exploration-rate schedule")
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 0]
success_rates = [random_results["success_rate"], eval_results["success_rate"]]
ax4.bar(["Random policy", "Q-learning"], success_rates, color=["tab:gray", "tab:blue"], alpha=0.8)
ax4.set_ylabel("Successful episodes (proportion)")
ax4.set_ylim(0, 1)
ax4.set_title("Evaluation success rate")
ax4.grid(True, axis="y", alpha=0.3)

ax5 = axes[1, 1]
all_q_values = [
    value
    for state_actions in agent.q_table.values()
    for value in state_actions.values()
]
ax5.hist(all_q_values, bins=30, alpha=0.75, edgecolor="black", color="skyblue")
ax5.set_xlabel("Q-value (expected discounted reward)")
ax5.set_ylabel("Number of state-action values")
ax5.set_title("Learned Q-value distribution")
ax5.grid(True, alpha=0.3)

ax6 = axes[1, 2]
last_rewards = training_history["episode_rewards"][-100:]
last_episodes = episodes[-len(last_rewards):]
ax6.plot(last_episodes, last_rewards, color="tab:blue", linewidth=1.5, label="Episode reward")
ax6.axhline(
    y=np.mean(last_rewards),
    color="tab:red",
    linestyle="--",
    label=f"Mean: {np.mean(last_rewards):.2f}",
)
ax6.set_xlabel("Episode")
ax6.set_ylabel("Episode reward (reward units)")
ax6.set_title("Final 100 training episodes")
ax6.legend()
ax6.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 60)
print("REINFORCEMENT LEARNING ANALYSIS")
print("=" * 60)

initial_rewards = training_history["episode_rewards"][:10]
final_rewards = training_history["episode_rewards"][-10:]
print("\n1. TRAINING SUMMARY")
print(f"   Episodes trained: {len(training_history['episode_rewards'])}")
print(f"   Mean reward over the first 10 episodes: {np.mean(initial_rewards):.2f}")
print(f"   Mean reward over the final 10 episodes: {np.mean(final_rewards):.2f}")
print(f"   Difference between these means: {np.mean(final_rewards) - np.mean(initial_rewards):+.2f}")
print(f"   State keys visited: {len(agent.q_table)}")
print(f"   Final exploration probability: {agent.epsilon:.4f}")

print("\n2. GREEDY-POLICY EVALUATION")
print(f"   Success rate: {eval_results['success_rate']:.1%}")
print(f"   Mean distance: {eval_results['mean_distance']:.2f} +/- {eval_results['std_distance']:.2f} distance units")
print(f"   Mean emissions: {eval_results['mean_emissions']:.2f} +/- {eval_results['std_emissions']:.2f} emissions units")
print(f"   Mean reward: {eval_results['mean_reward']:.2f} +/- {eval_results['std_reward']:.2f}")

print("\n3. DIFFERENCE FROM THE RANDOM VALID-ACTION BASELINE")
print(f"   Mean reward difference: {eval_results['mean_reward'] - random_results['mean_reward']:+.2f}")
print(f"   Mean distance difference: {eval_results['mean_distance'] - random_results['mean_distance']:+.2f} distance units")
print(f"   Mean emissions difference: {eval_results['mean_emissions'] - random_results['mean_emissions']:+.2f} emissions units")
print(f"   Success-rate difference: {(eval_results['success_rate'] - random_results['success_rate']) * 100:+.1f} percentage points")

print("\n4. Q-TABLE SUMMARY")
print(f"   Stored state-action values: {len(all_q_values)}")
if all_q_values:
    print(f"   Q-value range: [{min(all_q_values):.2f}, {max(all_q_values):.2f}]")
    print(f"   Mean Q-value: {np.mean(all_q_values):.2f}")
    print(f"   Q-value standard deviation: {np.std(all_q_values):.2f}")
else:
    print("   No state-action values were recorded.")

print("\n5. INTERPRETATION")
print("   Training rewards and rolling averages describe observed learning behavior; they do not establish convergence.")
print("   Evaluation summarizes repeated runs on this same fixed instance, not performance on unseen instances.")
print("   Reward, distance, emissions, and successful completion measure different aspects of performance.")
print("   Tabular Q-learning provides no guarantee that the learned policy is globally optimal.")

## Interpreting the results

The training plots show episode reward, episode length, the exploration schedule, and the distribution of stored Q-values. A rising reward trend may indicate that the agent is finding higher-return decisions in this instance, but reward variation is expected under exploration and does not by itself demonstrate convergence. The final-episode plot is a descriptive stability check, not a statistical guarantee.

The evaluation compares the greedy policy with a policy that selects uniformly from feasible actions. Read each measure separately: higher reward is preferred under the specified reward design, while distance and emissions are separate travel measures and successful completion indicates whether all customers were served before termination. A difference from the random baseline is not proof of optimality, and results from this one fixed instance do not establish generalization.

The agent uses a tabular Q-function and an \(\varepsilon\)-greedy policy. Its state includes location, battery, unvisited customers, and capacity used on the current depot trip. This explicit state representation makes the routing and resource constraints visible, while the number of possible customer subsets grows exponentially with the customer count. The example is therefore intended to demonstrate the learning process on a small problem rather than to provide a scalable routing system.

The reward weights and fixed rewards shape the policy learned by the agent. Changing them would define a different learning objective; the resulting policy should therefore be interpreted in relation to the reward function, not as an unqualified measure of environmental or operational quality. Extensions may add realistic constraints or evaluate additional fixed instances while retaining the same Q-learning approach.